# This code used for slicing big-tiles into smaller pices for image annotation

In [1]:
import rasterio
from rasterio.windows import Window
from PIL import Image
import numpy as np
import os

def geotiff_slicing(input_tiff_path, output_folder, output_size=(1024, 1024), to_jpg=False):
    """
    Slices a GeoTIFF image into smaller tiles and optionally converts the slices to JPEG.

    Parameters:
    - input_tiff_path: str, Path to the input GeoTIFF image.
    - output_folder: str, Path to the folder where output images will be saved.
    - output_size: tuple, Desired (width, height) of the output slices (default is 1024x1024).
    - to_jpg: bool, If True, converts the output slices to JPG format (default is False).

    Returns:
    - List of paths to the output images (either .tiff or .jpg).
    """

    # Open the GeoTIFF file
    with rasterio.open(input_tiff_path) as dataset:
        # Get the width and height of the image
        img_width = dataset.width
        img_height = dataset.height

        # Calculate the number of horizontal and vertical slices
        x_slices = img_width // output_size[0]
        y_slices = img_height // output_size[1]

        output_paths = []
        
        for i in range(y_slices):
            for j in range(x_slices):
                # Define the window for each slice
                window = Window(j * output_size[0], i * output_size[1], output_size[0], output_size[1])
                
                # Read the window from the GeoTIFF
                transform = dataset.window_transform(window)
                img_data = dataset.read(window=window)
                
                # Create the output file name
                output_filename = f"tile_{i}_{j}.tiff"
                output_path = os.path.join(output_folder, output_filename)

                # Write the slice to a new GeoTIFF file
                with rasterio.open(
                    output_path, 'w',
                    driver='GTiff',
                    height=output_size[1],
                    width=output_size[0],
                    count=dataset.count,
                    dtype=img_data.dtype,
                    crs=dataset.crs,
                    transform=transform
                ) as dest:
                    dest.write(img_data)
                
                # Convert to JPEG if required
                if to_jpg:
                    img = Image.fromarray(np.moveaxis(img_data, 0, -1))  # Convert to (H, W, C) format

                    # Check if the image is RGBA and convert it to RGB
                    if img.mode == 'RGBA':
                        img = img.convert('RGB')

                    output_jpg_path = output_path.replace('.tiff', '.jpg')
                    img.save(output_jpg_path, 'JPEG')
                    os.remove(output_path)  # Remove the .tiff if converting to .jpg
                    output_paths.append(output_jpg_path)
                else:
                    output_paths.append(output_path)

    return output_paths


In [2]:
# !pip install rasterio

In [3]:
# input_tiff = "m_3008501_ne_16_030_20211103.tif"
# output_dir = "output_tiles_1"
output_tile_size = (3000, 3000)  # Slice size: 512x512 pixels

# # Slice and convert to JPG
# output_paths = geotiff_slicing(input_tiff, output_dir, output_tile_size, to_jpg=True)
# print("Generated slices:", output_paths)


In [9]:
import glob
import os

def find_tif_images(folder_path):
    return glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)

output_tile_size = (3000, 3000)
folder_path = 'E:/Data/al_30cm_2021_transfer_1_of_2/31087/'
tif_images = find_tif_images(folder_path)
for tif_image in tif_images:
    new_path = tif_image.replace("E:/Data", "E:/NewData").replace(".tif", "")
    # print(tif_image)
    # print(new_path)
    os.makedirs(new_path, exist_ok=True)
    output_paths = geotiff_slicing(tif_image, new_path, output_tile_size, to_jpg=True)
